# 상태 기록과 다른 분기 재실행

완료된 실행의 중간 체크포인트에서 값을 바꾸어 원본을 보존한 새 분기를 만들 수 있는지 확인합니다.


## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.


In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11

Note: you may need to restart the kernel to use updated packages.


## 가격 계산 그래프 준비

가격 확정 뒤 할인율을 적용하는 두 노드로 분기 지점을 만듭니다.

상태 기록에는 현재 값뿐 아니라 단계별 `StateSnapshot`이 남습니다. `get_state_history(config)`는 최신 기록부터 반환하고, 각 snapshot의 `next`는 그 상태에서 다음에 실행할 노드를 나타냅니다. `next == ("discount",)`인 기록을 고르면 quote는 끝났지만 discount는 실행되기 전인 분기 지점이 됩니다.

`update_state(before_discount.config, {"rate": 0.2})`는 과거 기록을 수정해 덮어쓰지 않습니다. 그 체크포인트를 부모로 하는 새 체크포인트 설정을 반환합니다. `invoke(None, fork_config)`는 저장된 quote 결과를 사용하고 그 뒤의 discount부터 실행합니다. 원본 완료 config를 따로 보관하면 원본과 분기 결과를 각각 다시 읽을 수 있습니다.


In [2]:
# 가격·할인율·실행 기록을 담는 상태와 두 계산 노드를 정의합니다.
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

class State(TypedDict):
    base_price: int
    rate: float
    final_price: int
    trace: list[str]

def quote(state: State) -> dict:
    return {"trace": state["trace"] + ["quote"]}

def discount(state: State) -> dict:
    return {
        "final_price": int(state["base_price"] * (1 - state["rate"])),
        "trace": state["trace"] + [f"discount:{state['rate']}"]
    }

## 그래프 컴파일

두 노드를 순서대로 연결하고 메모리 체크포인터를 붙입니다.


In [3]:
# quote 다음 discount가 실행되도록 그래프를 연결하고 체크포인터를 붙입니다.
builder = StateGraph(State)
builder.add_node("quote", quote)
builder.add_node("discount", discount)
builder.add_edge(START, "quote")
builder.add_edge("quote", "discount")
builder.add_edge("discount", END)
graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "price-2403"}}

## 원본 실행과 기록 조회

`next`가 `discount`인 상태를 분기 기준으로 고릅니다.


In [4]:
# 할인율 0.1의 원본을 완료하고 next가 discount인 체크포인트를 고릅니다.
original = graph.invoke(
    {"base_price": 10_000, "rate": 0.1, "final_price": 0, "trace": []}, config
)
history = list(graph.get_state_history(config))
before_discount = next(item for item in history if item.next == ("discount",))
original_final_config = graph.get_state(config).config
print("original:", original)
print("history:", [(item.metadata.get("step"), item.next) for item in reversed(history)])
assert original["final_price"] == 9000

original: {'base_price': 10000, 'rate': 0.1, 'final_price': 9000, 'trace': ['quote', 'discount:0.1']}
history: [(-1, ('__start__',)), (0, ('quote',)), (1, ('discount',)), (2, ())]


## 할인율을 바꾼 분기 실행

선택한 체크포인트의 상태를 갱신한 뒤 `None`으로 이어서 실행합니다.

프롬프트나 임계값을 바꿔 후속 단계만 다시 평가할 때 같은 방식을 사용할 수 있습니다. 다만 분기 지점 뒤의 모델 호출과 API 요청은 다시 실행되므로 저장된 결과와 새 결과의 조건을 기록해야 합니다. reducer가 있는 상태 키를 update하면 값이 대체되지 않고 누적될 수 있다는 점도 확인합니다.

그래프의 time travel은 체크포인트에서 새 실행 경로를 만드는 기능이며 외부 세계의 시간을 되돌리지 않습니다. 이미 전송한 이메일이나 결제를 취소하지 않으므로 보상 작업이나 격리가 필요합니다. 이번 trace는 quote가 재실행되지 않은 상태 경로를 보여 줄 뿐 외부 부작용 복원을 검증하지 않습니다.


In [5]:
# 선택한 상태의 할인율만 0.2로 바꾸어 새 분기의 최종 가격을 계산합니다.
fork_config = graph.update_state(before_discount.config, {"rate": 0.2})
forked = graph.invoke(None, fork_config)
original_saved = graph.get_state(original_final_config)
fork_saved = graph.get_state(fork_config)
print("original final:", original_saved.values["final_price"])
print("fork final:", forked["final_price"])
print("fork trace:", forked["trace"])
print("checkpoint ids differ:", original_saved.config != fork_saved.config)
assert original_saved.values["final_price"] == 9000
assert forked["final_price"] == 8000

original final: 9000
fork final: 8000
fork trace: ['quote', 'discount:0.2']
checkpoint ids differ: True
